# 2. Develop and deploy: source and manifest preparation

**Pre-live authoring draft — not a complete participant lifecycle tutorial.**

Run this notebook from `beginner/` with a Python kernel. All current code cells
are offline and use the standard library. They inspect existing source, edit
a temporary source/test copy, render candidate manifests, and demonstrate
versioned image references in temporary storage. They do not execute JavaScript, run a CLI, build, push or deploy images.

The selected beginner workflow remains participant-executed build → push →
deploy in an organiser-prepared disposable lab. Organiser-mediated deployment
is not a substitute. The protected builder, registry permissions, authentication,
isolation and exact platform/template versions must be resolved and rehearsed
before executable live lifecycle cells can be added.


## 1. Locate source and helpers — offline

**Purpose:** connect function source to the manifest describing its image and route.
**Location:** this notebook's Python kernel, started in `beginner/`.
**Expected result:** the module path and a validated offline template.
**Recovery:** restart the kernel from `beginner/` if the module check fails.


In [ ]:
from pathlib import Path
import json
import sys
import tempfile

MODULE = Path.cwd().resolve().parent
if not (MODULE / "config/lab.example.json").is_file():
    raise RuntimeError("Open this notebook from the module's beginner directory.")
sys.path.insert(0, str(MODULE / "helpers"))
from lab_config import load_config
from manifests import render_manifest, write_manifest, TEMPLATE_REVISION

CONFIG = MODULE / "config/lab.example.json"
config = load_config(CONFIG, require_live=False)
print("Module:", MODULE)
print("Inspected template revision (not enforced by this helper):", TEMPLATE_REVISION)


## 2. Inspect the complete greeting implementation — offline

The developer controls input validation and the successful response. Review the
POST check, object check, name trimming and length bound. The handler does not
install a platform or choose its own resource placement.

**Checkpoint:** locate the response message and the `invalid_name` branch.
The supplied implementation is complete; this inspection is not a TODO exercise.
The package test command requires a separately validated Node.js environment.
Python does not execute or validate the JavaScript shown here.


In [ ]:
print((MODULE / "examples/greeting/handler.js").read_text(encoding="utf-8"))
package = json.loads((MODULE / "examples/greeting/package.json").read_text(encoding="utf-8"))
print("Declared runtime:", package["engines"])
print("Declared unit-test command:", package["scripts"]["test"])


## 3. Change source and test expectations — offline rehearsal

**Purpose:** distinguish a code change from a request-data change, and keep
tests aligned with an intentional response-contract update.
**Location:** this Python kernel; all writes are inside a temporary directory.

We change `Hello` to `Welcome` in a copied handler. First, the copied tests
still expect `Hello`: running them at this stage should fail the three
successful-response cases. We then update those expectations. The invalid
input and GET tests remain unchanged. This cell does not run either suite;
the expected failures and passes still require Node.js execution.

**Action:** run the complete cell below. Read the two diffs before the
temporary files are removed. No reference files or local lab configuration
are edited, and no dependencies are installed.
**Expected result:** one handler-line change, one test-expectation-line
change, and a candidate manifest pointing to the edited source copy.
**Checkpoint:** inspect `Welcome ${name}` in both diffs and the candidate
`handler` path. The existing renderer otherwise targets the reference
example, so an edited copy needs its own source path.
**Recovery:** if an anchor check fails, stop and compare the reference
handler and tests with the earlier inspection. Do not remove the checks.
Rerunning is safe: each run creates a fresh temporary directory.


In [ ]:
import difflib

original_message = "Hello ${name}, your function is live!"
revised_message = "Welcome ${name}, your function is live!"
reference_source = MODULE / "examples/greeting"
reference_files_before = {
    filename: (reference_source / filename).read_bytes()
    for filename in ("handler.js", "handler.test.js", "package.json")
}
source_before = reference_files_before["handler.js"].decode("utf-8")
tests_before = reference_files_before["handler.test.js"].decode("utf-8")
if source_before.count(original_message) != 1 or tests_before.count(original_message) != 1:
    raise RuntimeError("Expected exactly one greeting anchor in source and tests.")

with tempfile.TemporaryDirectory() as edit_directory:
    edited_source_directory = Path(edit_directory) / "greeting"
    edited_source_directory.mkdir()
    for filename, content in reference_files_before.items():
        (edited_source_directory / filename).write_bytes(content)

    source_after = source_before.replace(original_message, revised_message, 1)
    (edited_source_directory / "handler.js").write_text(source_after, encoding="utf-8")
    assert original_message in (edited_source_directory / "handler.test.js").read_text(encoding="utf-8")
    print("Checkpoint: source now says Welcome; tests still expect Hello.")
    print("Expected Node.js outcome at this stage: three success cases fail (not run here).")

    tests_after = tests_before.replace(original_message, revised_message, 1)
    (edited_source_directory / "handler.test.js").write_text(tests_after, encoding="utf-8")
    assert source_after.replace(revised_message, original_message, 1) == source_before
    assert tests_after.replace(revised_message, original_message, 1) == tests_before
    assert (edited_source_directory / "package.json").read_bytes() == reference_files_before["package.json"]

    for filename, before, after in (
        ("handler.js", source_before, source_after),
        ("handler.test.js", tests_before, tests_after),
    ):
        print("".join(difflib.unified_diff(
            before.splitlines(keepends=True), after.splitlines(keepends=True),
            fromfile="reference/" + filename, tofile="edited/" + filename,
        )))

    edited_candidate = json.loads(render_manifest(CONFIG, "greeting"))
    edited_name = config["functions"]["greeting"]
    edited_candidate["functions"][edited_name]["handler"] = str(edited_source_directory)
    edited_candidate["functions"][edited_name]["image"] = (
        config["registry_repository"] + "/" + edited_name + ":v2"
    )
    assert Path(edited_candidate["functions"][edited_name]["handler"]) == edited_source_directory
    assert (edited_source_directory / "handler.js").read_text(encoding="utf-8") == source_after
    print("Edited-source candidate (temporary, not built or deployed):")
    print(json.dumps(edited_candidate, indent=2))
    print("Node.js test command for a retained working copy: npm test")
    print("Expected after aligning tests: all 15 pass; not executed here.")

assert all((reference_source / filename).read_bytes() == content
           for filename, content in reference_files_before.items())
print("PASS: Only temporary source and test copies changed.")
print("Temporary edited source removed; its printed candidate is no longer usable.")


## 4. Render a reference-source candidate manifest — offline

**Purpose:** map one owned function to its source, template and versioned image.
**Expected result:** one `group-01-greeting` entry, with `node22` and tag `v1`.
The `.invalid` routes are deliberate placeholders, not a lab you can contact.

The renderer returns JSON-formatted YAML. Acceptance by the pinned CLI and the
absolute handler path remains a separate validation step. The template revision
printed earlier is provenance information: the renderer does not fetch, install
or enforce that revision. Function prefixes check consistency, not permissions.


In [ ]:
manifest_text = render_manifest(CONFIG, "greeting")
manifest = json.loads(manifest_text)
name = config["functions"]["greeting"]
assert set(manifest["functions"]) == {name}
assert manifest["functions"][name]["lang"] == "node22"
assert manifest["functions"][name]["image"].endswith(":" + config["image_tag"])
print(manifest_text)


## 5. Check safe manifest writing — offline, temporary files only

**Purpose:** prevent accidental replacement of an existing manifest.
**Expected result:** the file matches the rendered candidate and a second write
is refused. The temporary directory is removed at the end of the cell.
No local configuration or persistent manifest is created in the repository.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    candidate = Path(directory) / "greeting.local.json"
    write_manifest(CONFIG, "greeting", candidate)
    assert candidate.read_text(encoding="utf-8") == manifest_text
    try:
        write_manifest(CONFIG, "greeting", candidate)
    except FileExistsError:
        print("PASS: Existing candidate cannot be overwritten.")
    else:
        raise RuntimeError("Existing candidate was unexpectedly overwritten.")
print("Temporary candidate removed.")


## 6. Distinguish request changes from deployed code changes — offline

Sending `{"name": "Ada"}` and then `{"name": "Sam"}` changes request data;
it does not change function code. Changing the response implementation requires
a new image build, push, deployment/update and verification.

Below we change only an image tag in a temporary configuration. This demonstrates
reference construction, not a code update. A tag is a label and may be mutable;
an explicit tag alone does not prove that an image exists or pin its digest.


In [ ]:
updated_config = dict(config)
updated_config["image_tag"] = "v2"
with tempfile.TemporaryDirectory() as directory:
    temporary_config = Path(directory) / "lab.local.json"
    temporary_config.write_text(json.dumps(updated_config), encoding="utf-8")
    updated_manifest = json.loads(render_manifest(temporary_config, "greeting"))
old_image = manifest["functions"][name]["image"]
new_image = updated_manifest["functions"][name]["image"]
assert old_image != new_image
assert new_image.endswith(":v2")
print("Original candidate:", old_image)
print("Updated candidate:", new_image)
print("No source, image, registry or deployed function was changed.")


## 7. Keep your own working copy and run JavaScript tests

The temporary rehearsal above protects the reference files but removes your
edits. This exercise creates a retained working copy that you can edit in
JupyterLab. It needs no gateway, registry, container daemon or external API.

**Execution location:** a JupyterLab Bash terminal on the notebook host—not
this Python kernel and not your laptop terminal. Open a terminal and navigate
to the module directory `05-openfaas/`. Its contents must include `beginner/`,
`examples/`, and `config/`. The first check below stops if you are elsewhere.

**Prerequisite:** the organiser supplies Node.js 22 and npm on the notebook
host. Do not install Node.js, use sudo, or mount a Docker socket for this
exercise. If the tools are missing or the Node.js major version is not 22,
stop and ask the organiser to prepare the authoring runtime.

### 7.1 Prepare a retained source copy

Run this in the terminal from the module directory:

```bash
(
  set -euo pipefail
  test -f config/lab.example.json
  test -f examples/greeting/handler.test.js
  mkdir -p workspace
  if [ -e workspace/greeting ] || [ -L workspace/greeting ]; then
    printf 'STOP: workspace/greeting already exists; preserve your edits.\n' >&2
    exit 1
  fi
  cp -R -- examples/greeting workspace/greeting
  printf 'Created workspace/greeting; reference files are unchanged.\n'
  ls workspace/greeting
)
```

**Expected result:** `workspace/greeting/` contains `handler.js`,
`handler.test.js`, and `package.json`. The directory is ignored by Git;
ignored does not mean backed up or safe for secrets.
**Recovery:** if the copy already exists, inspect it and continue with it
rather than replacing it. If copying failed, ask for help inspecting the
partial directory before retrying. Do not run blanket cleanup commands.

### 7.2 Run the baseline tests

Still from the module directory, run:

```bash
(
  set -euo pipefail
  test -f config/lab.example.json
  cd workspace/greeting
  node --version
  npm --version
  node -e 'if (process.versions.node.split(".")[0] !== "22") { console.error("STOP: Node.js 22 is required."); process.exit(1); }'
  npm test
)
```

No `npm install` is required: this example uses Node.js built-in test and
assertion modules and declares no external dependencies.
**Expected baseline:** 15 tests pass, with no failures. These are handler
unit tests using a simulated context—not watchdog, HTTP or platform tests.
**Recovery:** missing tools require organiser assistance. If tests fail,
inspect the failures and compare your working copy with the reference.
A baseline failure must be resolved before making the next change.

### 7.3 Change the response, then observe test failures

In JupyterLab's file browser, open **your copy**:
`workspace/greeting/handler.js`. Change only the successful message:

```javascript
message: `Welcome ${name}, your function is live!`
```

Save the file. Leave `handler.test.js` unchanged and rerun the command from
section 7.2. **Expected result:** the three successful-response tests fail
because they still expect `Hello`; the twelve rejection tests pass. The
command should return a nonzero exit status. This deliberate failure shows
a changed response contract; it is not evidence of a deployment problem.

The text “your function is live” is response content inherited from the
example. A local test returning it does not mean a function was deployed.

### 7.4 Align the intended response contract

Open `workspace/greeting/handler.test.js`. In the successful-response loop,
change the expected message from `Hello` to `Welcome`. This one expectation
is shared by the valid-name, trimmed-name and 80-character-name cases.
Do not remove assertions or change the invalid-input or GET expectations.

Save and rerun section 7.2. **Expected result:** all 15 tests pass.
**Checkpoint:** explain why trimming and rejection behaviour are unchanged,
and why changing request data alone would not require this source edit.
If the result differs, inspect the named failing test instead of weakening it.

### 7.5 Retain and inspect your work

Your edited files remain in `workspace/greeting/` after the commands finish.
From the module directory, inspect the changes:

```bash
diff -u examples/greeting/handler.js workspace/greeting/handler.js
diff -u examples/greeting/handler.test.js workspace/greeting/handler.test.js
git status --short
```

Run these individually: `diff` returns status 1 when differences exist,
which is expected here. Each diff should show only the greeting-message
change. Workspace files should not appear in Git status.

**Handoff to the later platform sprint:** a build manifest for your changed
function must point its `handler` field to `workspace/greeting`, not to
`examples/greeting`. The existing renderer targets the reference example.
The `v2` image reference shown earlier does not build or deploy this copy.
Keep the copy for later exercises; executable build/push/deploy steps remain
separate and unvalidated. You do not need platform credentials here.


## 8. Live lifecycle — BLOCKED in this draft

No switch enables these actions here. Complete runnable participant cells will
be added only after the execution contract is documented and tested.

| Participant stage | Required verification before adoption |
|---|---|
| Build | Protected builder access, exact template and architecture, actual image creation and test evidence. Shrinkwrap is not an image build. |
| Push | Correct built-image environment, scoped registry credentials, successful command status and registry evidence. |
| Deploy/update | Supported edition and registry behaviour, assigned-function permissions, correct image reference and negative cross-group tests. |
| Readiness | A validated readiness mechanism with bounded waiting; printed deployment progress is insufficient. |
| Invoke | Verified TLS/access, actual gateway/runtime responses and expected code version. |
| Cleanup | Only owned exercise functions through validated scoped permissions; never remove the prepared function or platform. |

The inspected CLI candidate uses separate build, push and deploy operations.
Its tested push failure does not establish successful registry access. The
inspected CE provider's anonymous-image restriction is an adoption blocker for
any proposed private-image path until actual behaviour and permitted use are
resolved. Do not work around this by publishing secrets or granting unrestricted
registry access. Participants must not receive shared-host Docker sockets or
shared-platform administrator credentials.


## 9. Review and handoff

- Which manifest field points to source, and which points to the runtime image?
- Why does rendering a manifest not demonstrate a successful image build?
- Why do the three successful-response tests need updating after the message changes?
- Why must the candidate handler path point to the edited source?
- What evidence shows that tag `v2` contains changed code?
- Why must registry permissions and runtime image retrieval both be tested?

**Current limits:** Python cells check file changes, not JavaScript behaviour.
The terminal exercise provides Node.js test commands for a retained copy;
these require the organiser-supplied runtime and are not run by notebook
Run All. No build/push/deploy/readiness commands are implemented.
It is an offline preparation draft, not delivery-ready coverage of these outcomes.
Notebook 3 will explore supplied weather data and bounded-traffic observations;
monitoring and cleanup will retain their live validation gates.
